# 00 · Introduction: CrewAI

Welcome. In this tutorial you will build two working agents with CrewAI: **ShipFast**, a support agent
that looks up orders and issues refunds (and asks a person before any refund over $100), and a **research
agent** that searches, drafts an answer and critiques its own draft. You will get there one idea at a time,
from a single model call in notebook 01 to the finished agents in notebook 08.

First, though, it helps to know what CrewAI is and how it thinks about agents, because that explains
most of the design choices you will meet along the way. At the end you will check that your setup works.

## What it is

**CrewAI** is an open-source Python framework created by João Moura, with its home at crewai.com. Its README
describes it as "a lean, lightning-fast Python framework built entirely from scratch, completely independent
of LangChain or other agent frameworks", offering "both high-level simplicity and precise low-level
control". It names two halves:

- **Crews**, which "optimize for autonomy and collaborative intelligence": teams of agents that each have a
  role.
- **Flows**, which "enable granular, event-driven control, single LLM calls for precise task orchestration
  and supports Crews natively".

The company also sells a hosted suite (the "CrewAI AMP Suite", with a cloud control plane for tracing and
deployment). This tutorial uses only the open-source library, locally.

> **Versions.** The quotes are from CrewAI's README. Everything in this tutorial uses CrewAI 1.15.34;
> newer versions may differ in the details.

## The core idea

![CrewAI: the core idea](images/crewai_core_idea.svg)

CrewAI's idea is that agents are **colleagues with roles**, and code gives them structure:

- An **Agent** is described like a person: a *role*, a *goal* and a *backstory*, plus tools and a model. It
  runs its own tool loop.
- A **Task** is a piece of work with an expected output, assigned to an agent.
- A **Crew** runs agents and tasks with a *process*: sequential, or hierarchical with a manager agent that
  delegates.
- A **Flow** is plain Python methods wired with `@start`, `@listen` and `@router`, sharing a typed state.
  Flows are where structure, persistence (`@persist`) and pauses for a person (`@human_feedback`) live.

CrewAI's own advice, which this tutorial follows: start with a Flow for structure, and call a Crew or an
agent inside the steps that need autonomy.

## How the course framed it

In Module 4 of the Agentic AI Crash Course, the "LangGraph vs CrewAI" slide sums CrewAI up like this: its mental model is "a team of
role-playing agents, inside Flows"; control is "higher level: describe roles and tasks; Flows add
structure"; its strengths are "fast to prototype agent teams; readable role definitions"; and the thing to
watch for is "less control inside a crew; role prompts can hide behaviour". A good first project, the slide
suggests, is "a research or content team", which is close to the research agent you will build.

## What we found building with it

We built the same two agents in LangGraph, CrewAI and Strands, with the same tools, data and local model,
and scored them the same way. Here is how CrewAI did:

- Results here: ShipFast 7 of 8 and research 9 of 10, using the most tokens of the three.
- Running fully locally took the most configuration. CrewAI defaulted to the cloud in four places
  (anonymous telemetry, OpenAI embeddings for memory, an OpenAI model inside `Memory`, and an OpenAI model
  inside `@human_feedback`); `crew_agents` sets all of them to local or off.
- It reaches Ollama through the OpenAI-compatible API, which is why the shared model carries its settings
  itself, and it had to be told the real context size.
- The approval pause sits **between Flow steps** and resumes only from saved state, so a refund becomes its
  own step. In a notebook, use the `_async` versions of `kickoff`.

Treat these as a first look rather than a verdict: each case ran once, with a small model on one laptop.
The full side-by-side is in `COMPARISON.md` at the repo root.

## The concepts, and where you meet them

| Concept | Notebook |
|---|---|
| `LLM`, `call`, structured output with `response_model` | 01 |
| Tools (`tool`), raw tool calls, `available_functions` | 02 |
| `Agent`, `Task`, `Crew`, `max_iter`, `kickoff_async` | 03 |
| Conversation history; `Memory` with a local embedder | 04 |
| `Flow`: `@start`, `@listen`, `@router`, state | 05 |
| `@human_feedback`, providers, `@persist`, resuming | 06 |
| The event bus and listeners for metrics | 07 |
| Putting it together: ShipFast and the research agent | 08 |

## How this tutorial works

![The path through every tutorial](images/roadmap.svg)

- **Same path in every framework.** The LangGraph, CrewAI and Strands tutorials have the same notebooks in
  the same order, so you can read one notebook in two frameworks side by side. Each notebook ends with
  "How the others do this".
- **Two running examples.** **ShipFast**, a customer-support agent for a made-up shop (orders, refund
  policy, a human approval for refunds over $100, memory across turns), and a **research agent** that answers
  questions about a made-up company from fixed articles, with a critique loop.
- **One shared contract.** The data, tools, instructions, scoring and evaluation runner live once, in
  `contracts/`; this tutorial wraps them in CrewAI. Its reusable code lives in `crew_agents/`, and both the
  notebooks and `run_eval.py` import it, so the code you read is the code that was tested.
- **One local model.** `qwen3-agents` (Qwen 3 8B) on Ollama, with fixed settings, so differences come from
  the framework. No API key is needed.
- **Real outputs, mistakes included.** Every notebook has been run, and the notes describe what actually
  happened, including where the small model got things wrong. Those mistakes are part of the lesson. When
  you run a notebook yourself, expect the details to differ a little.

## Setup check

You need Ollama running (`ollama serve`) and the shared model created once
(`ollama create qwen3-agents -f contracts/ollama/Modelfile`; see the README). Pick this tutorial's kernel
(`.venv/bin/python` in this tutorial's folder) and run the cell below.

In [1]:
import crew_agents  # first: switches CrewAI's telemetry and console panels off (see below)
from importlib.metadata import version

from agent_contracts.harness import load_settings, verify_model

verify_model()
print("model:", load_settings().model)
print("crewai", version("crewai"))

model: qwen3-agents
crewai 1.15.34


If you see the model name and the versions, you are all set. Next up: **01 · The model**, where you make
your first call.